# MMOT ДЗ №1 — инференс LLM на чувашском

Ноутбук **демонстрации инференса**. Обучение — отдельно: пайплайн в `scripts/` (`extract_corpus.py` → `train_tokenizer.py` → `tokenize_data.py` → `train.py`), детали в `README.md`.

Запуск:
- **локально** — чекпоинт `checkpoints/base/best.pt` и токенизатор `tokenizer/bpe-8192.json` находятся автоматически по структуре репозитория (из корня или из `notebooks/`); при наличии CUDA/MPS инференс идёт на ускорителе, иначе CPU;
- **в Colab** — файлов нет, ноутбук сам предложит загрузить `best.pt` и `bpe-8192.json`.

Модель: decoder-only GPT, 6,82 млн параметров (d256/L6/H4), RoPE, tied embeddings, контекст 256 токенов (~800 символов). Претрейн с нуля на cvwiki (85 МБ текста).

В конце — бенчмарк: фактовые промпты с перефразировками и автоматической проверкой ключевого слова в ответе.

In [1]:
import os


def find_file(candidates):
    """Ищет файл по кандидатам путей — репозиторий можно открыть из корня или из notebooks/."""
    for rel in candidates:
        for root in (".", "..", "../.."):
            path = os.path.join(root, rel)
            if os.path.exists(path):
                return path
    return None


CKPT = find_file(["checkpoints/base/best.pt", "best.pt"])
TOKENIZER = find_file(["tokenizer/bpe-8192.json", "bpe-8192.json"])

if CKPT is None or TOKENIZER is None:  # Colab: файлы репозитория отсутствуют
    from google.colab import files  # noqa
    print("Загрузите checkpoints/base/best.pt и tokenizer/bpe-8192.json:")
    for _ in range(2):
        files.upload()
    CKPT, TOKENIZER = "best.pt", "bpe-8192.json"
assert os.path.exists(CKPT) and os.path.exists(TOKENIZER), "файлы не найдены"

try:
    import tokenizers
except ImportError:
    %pip install -q tokenizers

import torch

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__}, device: {device}")
print(f"ckpt: {CKPT}\ntokenizer: {TOKENIZER}")

torch 2.6.0, device: mps
ckpt: ../checkpoints/base/best.pt
tokenizer: ../tokenizer/bpe-8192.json


In [2]:
# Модель: локально импортируется из scripts/train.py (единый источник правды),
# в Colab — используется копия классов ниже.
import sys

TRAIN_PY = find_file(["scripts/train.py"])
if TRAIN_PY:
    sys.path.insert(0, os.path.dirname(TRAIN_PY))
    from train import TinyGPT
    print("TinyGPT импортирован из", TRAIN_PY)
else:
    import math
    import torch.nn as nn
    import torch.nn.functional as F

    class RoPE(nn.Module):
        def __init__(self, dim: int, max_seq: int):
            super().__init__()
            inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
            t = torch.arange(max_seq).float()
            freqs = torch.outer(t, inv_freq)
            self.register_buffer("cos", freqs.cos(), persistent=False)
            self.register_buffer("sin", freqs.sin(), persistent=False)

        @staticmethod
        def rotate_half(x):
            x1, x2 = x.chunk(2, dim=-1)
            return torch.cat((-x2, x1), dim=-1)

        def rotate(self, q, k):
            T = q.shape[2]  # срез под текущую длину — иначе broadcast ломает генерацию
            cos = torch.cat((self.cos, self.cos), dim=-1)[None, None, :T]
            sin = torch.cat((self.sin, self.sin), dim=-1)[None, None, :T]
            return q * cos + self.rotate_half(q) * sin, k * cos + self.rotate_half(k) * sin

    class CausalSelfAttention(nn.Module):
        def __init__(self, d_model, n_head, block_size, swa_window):
            super().__init__()
            self.n_head, self.d_head = n_head, d_model // n_head
            self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
            self.proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RoPE(self.d_head, block_size)
            self.swa_window = swa_window
            if swa_window:
                mask = torch.ones(block_size, block_size, dtype=torch.bool).tril()
                band = torch.ones(block_size, block_size, dtype=torch.bool).triu(-swa_window)
                self.register_buffer("mask", (mask & band).view(1, 1, block_size, block_size), persistent=False)

        def forward(self, x):
            B, T, C = x.shape
            q, k, v = self.qkv(x).split(C, dim=2)
            q = q.view(B, T, self.n_head, self.d_head).transpose(1, 2)
            k = k.view(B, T, self.n_head, self.d_head).transpose(1, 2)
            v = v.view(B, T, self.n_head, self.d_head).transpose(1, 2)
            q, k = self.rope.rotate(q, k)
            if self.swa_window:
                att = F.scaled_dot_product_attention(q, k, v, attn_mask=self.mask[:, :, :T, :T])
            else:
                att = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            return self.proj(att.transpose(1, 2).reshape(B, T, C))

    class Block(nn.Module):
        def __init__(self, d_model, n_head, block_size, swa_window):
            super().__init__()
            self.ln1 = nn.LayerNorm(d_model)
            self.attn = CausalSelfAttention(d_model, n_head, block_size, swa_window)
            self.ln2 = nn.LayerNorm(d_model)
            self.mlp = nn.Sequential(
                nn.Linear(d_model, 4 * d_model, bias=False), nn.GELU(),
                nn.Linear(4 * d_model, d_model, bias=False),
            )

        def forward(self, x):
            return x + self.mlp(self.ln2(x + self.attn(self.ln1(x))))

    class TinyGPT(nn.Module):
        def __init__(self, vocab_size, d_model, n_layer, n_head, block_size, swa_window=0, tie_emb=True):
            super().__init__()
            self.block_size = block_size
            self.tok_emb = nn.Embedding(vocab_size, d_model)
            self.blocks = nn.ModuleList(Block(d_model, n_head, block_size, swa_window) for _ in range(n_layer))
            self.ln_f = nn.LayerNorm(d_model)
            self.head = nn.Linear(d_model, vocab_size, bias=False)
            if tie_emb:
                self.head.weight = self.tok_emb.weight
            self.apply(self._init)
            for name, p in self.named_parameters():
                if name.endswith("proj.weight") or name.endswith("mlp.2.weight"):
                    nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

        @staticmethod
        def _init(m):
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            x = self.tok_emb(idx)
            for blk in self.blocks:
                x = blk(x)
            logits = self.head(self.ln_f(x))
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)).float(), targets.view(-1))
            return logits, loss

        @torch.no_grad()
        def generate(self, idx, max_new, temperature=0.8, top_k=50):
            for _ in range(max_new):
                idx_cond = idx[:, -self.block_size:]
                logits, _ = self(idx_cond)
                logits = logits[:, -1, :] / max(temperature, 1e-6)
                if top_k:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float("inf")
                idx = torch.cat([idx, torch.multinomial(F.softmax(logits, dim=-1), 1)], dim=1)
            return idx

    print("TinyGPT: встроенная копия (Colab без репозитория)")

TinyGPT импортирован из ../scripts/train.py


In [3]:
from tokenizers import Tokenizer

ck = torch.load(CKPT, map_location=device, weights_only=True)
a = ck["args"]
tok = Tokenizer.from_file(TOKENIZER)
model = TinyGPT(tok.get_vocab_size(), a["d_model"], a["n_layer"], a["n_head"],
                a["block_size"], a.get("swa_window", 0))
model.load_state_dict(ck["model"])
model.to(device).eval()
print(f"шаг обучения: {ck['step']}, val_loss: {ck.get('val_loss', float('nan')):.4f}")
print(f"параметров: {sum(p.numel() for p in model.parameters()):,}")

шаг обучения: 9295, val_loss: 3.1353
параметров: 6,822,400


In [4]:
@torch.no_grad()
def gen(prompt, n=200, temperature=0.8, top_k=50):
    dev = next(model.parameters()).device
    ids = tok.encode(prompt).ids
    x = torch.tensor([ids], device=dev)
    out = model.generate(x, n, temperature=temperature, top_k=top_k)[0].tolist()
    return tok.decode(out)

## 1. Свободные продолжения

In [5]:
for prompt in ["Чăваш Ен —", "Шупашкар хули", "Чăваш чěлхи", "Раççей"]:
    print(f"\n=== {prompt!r} (T=0.8, top-k=50) ===")
    print(gen(prompt, 150))


=== 'Чăваш Ен —' (T=0.8, top-k=50) ===


Чăваш Ен — Чăваш Енĕн Патăръел районĕн администрациллĕ-территориллĕ виçи. Администраци центрĕ — Сосновка ялĕ. Тавралăхĕ Халăх йышĕ ял тытăмĕ 2010-мĕш çулхи çыравпа ялта 117 çын, çав шутра 457 арçын, 415 хĕрарăм пурăннă.Истори Тутарстанти Пăва уесĕн, Нурăс вулăсне, 1920 çулхи юпа уйăхĕн 5 хыççăн Пăва уесне кĕнĕ. 1920 çулхи юпа уйăхĕн 5 хыççăн Патăръел уесне кĕнĕ. 1926 çулхи юпа уйăхĕн 24 хыççăн Патăръел уесне кĕнĕ. Ку пĕтĕм тапхăрта ял Шăмăршă вулăсĕнче шутланнă. 1927 çулхи юпа уйăхĕн 1 хыççăн Патăръел районне, 1935 çулхи пуш уйăхĕн 1 хыççăн Патăръел районне, 1939 çулхи пуш уйăхĕн 17 хыççăн Первомайски районне кĕнĕ. 1959 çулхи утă уйăхĕн 14 хыççăн хальхи вăхăт таран Патăръел районне кĕрет. Паллă çынсем Пыр

=== 'Шупашкар хули' (T=0.8, top-k=50) ===


Шупашкар хули, Шупашкар хули.Халăх йышĕ, ял тытăмĕ Муниципаллă йĕркелĕвĕсем: Мулкач урамĕсем Çӳлти Хура Пашикасси (Шупашкар районĕ) Каранкасси (Шупашкар районĕ) Анатри Хура Пашикасси (Йĕкĕт ял тăрăхĕ) Çирĕклĕ (Шăмăршă районĕ) Уйкасси (Тăвай районĕ)Паллă çынсем Денисов Владимир Прокопьевич, çыравçă.Асăрхавсем Каçăсем Категори:Чăваш Енри Шупашкар районĕн ялĕсем{{Театр ĕçченĕ  |Ят = Светлана Алтышев |Тăван ят =  |Ӳкерчĕк =  |Анлăшĕ = 260px |Ӳкерчĕке ăнлантарни = |Алпусни =  |Çуралнă вăхăт =  |Çуралнă вырăн = |Вилнĕ вăхăт =  |Вилнĕ вырăн =

=== 'Чăваш чěлхи' (T=0.8, top-k=50) ===


Чăваш чěлхи юрăсене Чăваш Республикинче юрăсене шăрантарма пулăшакан, унăн пултарулăхне çаксене пĕлтерме пултарать: Чăваш Республикин чăвашсен халăх юррисене чăваш чĕлхинче манăçа, морфологире, ташăсенче, халăх юррисене тиркевте усă кураççĕ.Чăвашла юрăсем Чăваш Республикин халăх артисчĕ (1996), Чăваш Республикин тава тивĕçлĕ артисчĕ (1998), Чăваш Республикин халăх артисчĕ (2002). Чăваш Республикин халăх артисчĕ (1998), Чăваш Республикин тава тивĕçлĕ артисчĕ (2001).Чăваш Республикин тава тивĕçлĕ культура ĕçченĕ (2011). Чăваш Ен тава тивĕçлĕ артисчĕ (1998), Раççей Федерацин тава тивĕçлĕ артисчĕ (2013). Чăваш Республикин тава тивĕçлĕ артисчĕ. Чăваш Республикин тава тивĕçлĕ ӳнер ĕçченĕ (1996).Чăваш Республикин тава тивĕçлĕ артисчĕ (1993), Чăваш Республикин тава тивĕçлĕ

=== 'Раççей' (T=0.8, top-k=50) ===


Раççей патшалăх шыв реестрĕн хыпарĕпе Тури Обь шыв бассейн округĕне кĕрет. Шыв хуçăлах участокĕ- Обь Нефтеюганск хулинчен Иртыш кĕриччен. Юханшывăн кĕçĕн бассейнĕ — Вах, юханшыв бассейнĕ — Тури Обь, Иртыш кĕриччен.Шыв ресурсĕсен федераци агентстви хатĕрленĕ РФ территорин шыв хуçалăх районĕсен геоинформаци системин хыпарĕпе : Патшалăх шыв реестрĕнчи шыв объекчĕн кочĕ — 13011100212115200047892 Гидрологи тĕпчевленӳ кочĕ — 115204789 Бассейн кочĕ — 13.01.11.002 Том номерĕ — 15 Кăларăм — 2Асăрхавсем Каçăсем РФ çутçанталăк ресурсĕсемпе экологи министерстви Категори:Хантă-Манси автономи тăрăхĕн юханшывĕсемКимовка — Раççей территоринчи юханшыв. Хантă-Манси автономи тăрăхĕ — Югра территорипе юхать. Юханшыв Ким юханшывăн сулахай çыранпе вăрринчен 183 км юхса кĕрет. Юханшыв тăршшĕ 17 км.Шыв реестрĕн хыпарĕсем Раççей патшалăх


## 2. Влияние температуры

In [6]:
for t in [0.3, 0.6, 0.9, 1.2]:
    print(f"\n=== T={t} ===")
    print(gen("Чăваш Ен —", 100, temperature=t))


=== T=0.3 ===


Чăваш Ен — Чăваш Енре вырнаçнă чăваш ялĕсем.Категори:Чăваш ялĕсемКукай, — Чăваш Енĕн Вăрнар районĕнчи ял.Тавралăхĕ Халăх йышĕ, ял тытăмĕ Кун-çулĕ Паллă çыннисем Вуламалли Каçăсем Категори:Вăрнар районĕн ялĕсемКукай, — Чăваш Енĕн Вăрнар районĕнчи ял. Тавралăхĕ Халăх йышĕ, ял тытăмĕ Кун-çулĕ Паллă çыннисем Асăрхавсем Вуламалли Каçăсем Категори:Вăрнар районĕн ялĕсемКукай, — Чăваш Енĕн Вăрнар районĕнчи ял. 

=== T=0.6 ===


Чăваш Ен — Чăваш Республикин Шупашкар районĕ Чăваш Ен — Чăваш Енĕн Шупашкар районĕ Чăваш Ен — СССР Категори:Чăваш Ен Категори:Пĕрлĕх ĕçченĕсем Категори:Чăваш АССР тава тивĕçлĕ ял хуçалăх ĕçченĕсем Категори:Чăваш АССР тава тивĕçлĕ ĕçченĕсем Категори:Чăваш АССР тава тивĕçлĕ артисчĕсем Категори:Чăваш АССР тава тивĕçлĕ артисчĕсем Категори:Чăваш Республикин тава тивĕçлĕ артисчĕсем Категори:Чăваш АССР тава тивĕçлĕ артисчĕсем Категори:Чăваш АССР тава тивĕçлĕ артисчĕсем Категори:Пайăр çынсем, алфавитпа Категори:Чăваш Енĕн паллă çыннисем Категори:РФ тава тивĕçлĕ артисчĕсем Категори:Чăваш АССР халăх

=== T=0.9 ===


Чăваш Ен — Тутарстанри Çĕпрел районĕ — Чăваш Енĕн Тутарстанăн Çĕпрел районĕ — Тамбов облаçĕн çур.) — чăваш çыравçи, сăвăçи (сÿлĕм). СССР ÇП пайташĕ (1974 ç.)Биографи Вуламалли Научный отчет по проекту "Чуваши Приволжского округа". Шупашкар, 2002. 192 с. Паллă çынсем Баганов Филипп Иванович (1883-1944) - Совет Союзĕн Геройĕ, СССР патшалăх думин депутачĕ (1985). Краснофлотец А.М

=== T=1.2 ===


Чăваш Ен — Чĕмпĕр облаçĕн районĕсем. Тутарстанти Чĕмпĕр облаçĕ Тутарстан, Чĕмпĕр облаçĕн Çĕпрел районĕ тата Чĕмпĕрской районĕнчи чăваш ялĕсен центрĕ, хальхи Чĕмпĕрте - Чĕмпĕр кĕпĕрнине кĕнĕ Етĕрне районĕнчи чăваш. Ун хыççăн Чĕмпĕр облаçĕнче шутланнă.1878 çулта кунта 92 вулăсран 175 кил пулнă: Хусан, Тутаркасси, Атăк, Элмес, Хĕрлĕ Çар, Сăрă , Хура Кӳлĕри ял пур.1878 çулта Самар кĕпернийĕн Хучтавра Культура районĕ никĕсленĕ. 1884


## 3. Бенчмарк: факты и перефразировки

Требование ДЗ — показать, что модель отвечает на фактовые вопросы **и сохраняет ответ при разных формулировках**. Модель — base-LM, обученная на вики-статьях, поэтому «вопрос» задаётся промптом-зачином в энциклопедическом стиле. Генерация при низкой температуре (T=0.3, top-k=20), ответ-продолжение автоматически проверяется на ключевое слово.

Группы:
- **Онегин → Пушкин**: связывает ли модель произведение с автором;
- **Шупашкар ↔ Чăваш Ен**: устойчивость географической ассоциации при трёх формулировках;
- **Атăл = юханшыв**: относит ли Волгу к рекам;
- **негативный контроль**: вопросительная форма («Кам çырнă …?») — вопросных предложений в вики-корпусе нет, ожидаем отказ.

In [7]:
import pandas as pd

GROUPS = [
    ("Онегин → Пушкин", ["Пушкин"], [
        "«Евгений Онегин» поэмăна çырнă",
        "«Евгений Онегин» поэмине кам çырнă",
    ]),
    ("Шупашкар ↔ Чăваш Ен", ["Чăваш"], [
        "Шупашкар —",
        "Шупашкар хули —",
        "Шупашкар — Чăваш Енĕн",
    ]),
    ("Атăл = юханшыв", ["юханшыв"], [
        "Атăл юханшыв —",
        "Атăл — Раççейри чи пысăк",
    ]),
    ("негативный контроль: вопрос с «?»", ["Пушкин"], [
        "Кам çырнă «Евгений Онегин»?",
    ]),
]

# Бенчмарк гоняется на CPU с фиксированным seed — воспроизводимо на любом железе
# (RNG-сэмплирование на MPS/CUDA даёт другие выборки).
model.to("cpu")
rows = []
for group, kws, prompts in GROUPS:
    for p in prompts:
        torch.manual_seed(42)
        cont = gen(p, n=48, temperature=0.3, top_k=20)
        answer = cont[len(p):].strip() if cont.startswith(p) else cont
        found = any(k.lower() in cont.lower() for k in kws)
        rows.append({
            "группа": group,
            "промпт": p,
            "продолжение (обрезано)": answer[:120],
            "ключевое слово": "/".join(kws),
            "найдено": "✓" if found else "✗",
        })
model.to(device)

df = pd.DataFrame(rows)
with pd.option_context("display.max_colwidth", 55, "display.width", 200):
    display(df)

print("\nКонсистентность по группам (доля формулировок с ключевым словом):")
print(df.groupby("группа")["найдено"].agg(lambda s: f"{(s == '✓').sum()}/{len(s)}"))

,группа,промпт,продолжение (обрезано),ключевое слово,найдено
0,Онегин → Пушкин,«Евгений Онегин» поэмăна çырнă,".Сюжет Рольсенче Николай Кравцов — Маша, почтальон-...",Пушкин,✗
1,Онегин → Пушкин,«Евгений Онегин» поэмине кам çырнă,.«Икĕ çул» (П. П. Пушкинăн «Песня о боевых дней» пь...,Пушкин,✓
2,Шупашкар ↔ Чăваш Ен,Шупашкар —,"Шупашкар, Шупашкар, Шупашкар, 2021.Асăрхавсем Каçăс...",Чăваш,✓
3,Шупашкар ↔ Чăваш Ен,Шупашкар хули —,Чăваш Республикин Шупашкар хули.Кун-çулĕ 1927 çулче...,Чăваш,✓
4,Шупашкар ↔ Чăваш Ен,Шупашкар — Чăваш Енĕн,"Шупашкар районĕнчи ял. Халăх йышĕ, ял тытăмĕ 2010-м...",Чăваш,✓
5,Атăл = юханшыв,Атăл юханшыв —,"Раççей территоринчи юханшыв. Томск облаçĕ, Краснояр...",юханшыв,✓
6,Атăл = юханшыв,Атăл — Раççейри чи пысăк,"хула.Тавралăхĕ Хула — Мускав, Мускав, Мускав, Муска...",юханшыв,✗
7,негативный контроль: вопрос с «?»,Кам çырнă «Евгений Онегин»?,.Асăрхавсем Категори:ССРП драма-фильмĕсем Категори:...,Пушкин,✗



Консистентность по группам (доля формулировок с ключевым словом):
группа
Атăл = юханшыв                       1/2
Онегин → Пушкин                      1/2
Шупашкар ↔ Чăваш Ен                  3/3
негативный контроль: вопрос с «?»    0/1
Name: найдено, dtype: object


## Выводы по бенчмарку

- **Шупашкар ↔ Чăваш Ен — 3/3**: все три формулировки дают продолжения про Чувашию («Чăваш Республикин…», «Чăваш Ен паллă çыннисем») — ассоциация устойчива к перефразировке.
- **Онегин → Пушкин — 1/2**: формулировка «поэмине кам çырнă» даёт продолжение с «П. П. Пушкинăн» — модель связывает произведение с автором; декларативный зачин «поэмăна çырнă» уводит в шаблон статьей о фильмах — знание факта неустойчиво.
- **Атăл = юханшыв — 1/2**: с явным словом «юханшыв» модель корректно продолжает «Раççей территоринчи юханшыв», без подсказки называет Волгу городом — при 6,8 млн параметров факты держатся только при явном контексте.
- **Негативный контроль — 0/1 (ожидаемо)**: на вопрос с «?» модель выдаёт вики-шаблонный текст без ответа — в корпусе вопросных форм нет, что подтверждает, что «ответы» обусловлены корпусом, а не вопросным форматом.

Итог: связный чувашский текст и устойчивые тематические ассоциации — есть; точные факты — только при формулировке, близкой к обучающему распределению.